# Review Usability Evaluation Reports
This notebook allows you to browse and review the pre-generated usability evaluation reports without requiring a Google AI API Key. It loads the results of the evaluation approach presented in the paper.

The results are divided into three sections, representing the pipeline stages:
1. **Heuristic Evaluation**: Raw findings from the initial MLLM evaluation for each task and heuristic.
2. **Aggregated Report**: Consolidated issues after merging overlapping findings across heuristics.
3. **Severity Ranking**: The final prioritized list of usability problems re-ranked by the MLLM.


In [1]:
import json
import os
import pandas as pd
from IPython.display import display, Markdown

# Choose the dataset to review
DATA_PATH = "../data/KnowledgeCheckR/"
# DATA_PATH = "../data/EventHelpR/"

REPORTS_PATH = os.path.join(DATA_PATH, "reports")


## 1. Heuristic Evaluation
This section shows the raw results of the heuristic evaluation, where each heuristic was evaluated independently for each task.


In [2]:
# Load all evaluation_*.json files
evaluation_files = sorted([f for f in os.listdir(REPORTS_PATH) if f.startswith("evaluation_") and f.endswith(".json")])
all_evaluation_issues = []

for file in evaluation_files:
    with open(os.path.join(REPORTS_PATH, file), "r") as f:
        data = json.load(f)
        for entry in data:
            heuristic = entry.get("heuristic", "Unknown")
            task_id = entry.get("id", "Unknown").split("_")[0:3]
            task_id = "_".join(task_id) if isinstance(task_id, list) else task_id
            for issue in entry.get("issues", []):
                issue['source_file'] = file
                issue['heuristic'] = heuristic
                issue['task_id'] = task_id
                all_evaluation_issues.append(issue)

df_eval = pd.DataFrame(all_evaluation_issues)
if not df_eval.empty:
    display(Markdown(f"### Raw Evaluation Issues for {DATA_PATH.split('/')[-2]}"))
    # Grouping by heuristic to show counts
    display(Markdown("#### Issues Count by Heuristic"))
    display(df_eval.groupby('heuristic').size().to_frame('count').sort_values('count', ascending=False))
    
    display(Markdown("#### Raw Issues"))
    display(df_eval[['task_id', 'severity', 'heuristic', 'description']])
else:
    print("No raw evaluation reports found.")


### Raw Evaluation Issues for KnowledgeCheckR

#### Issues Count by Heuristic

,count
heuristic,
Aesthetic and Minimalist Design,12
Visibility of System Status,12
Match between System and Real World,12
Error Prevention,11
Consistency and Standards,10
Help and Documentation,9
Recognition Rather than Recall,8
Flexibility and Efficiency of Use,8
User Control and Freedom,8


#### Raw Issues

,task_id,severity,heuristic,description
0,KCR_P1_T1,Major Issue,Visibility of System Status,"After registering, the system redirects to a l..."
1,KCR_P1_T1,Minor Issue,Visibility of System Status,When the user enters incorrect login credentia...
2,KCR_P1_T1,Minor Issue,Match between System and Real World,"The term ""CheckR App"" is used throughout the i..."
3,KCR_P1_T1,Minor Issue,User Control and Freedom,After attempting to log in with incorrect cred...
4,KCR_P1_T1,Minor Issue,Consistency and Standards,"After registering, the user is redirected to a..."
...,...,...,...,...
91,KCR_P2_T2,Minor Issue,Aesthetic and Minimalist Design,The progress bar is quite large and visually p...
92,KCR_P2_T2,Minor Issue,Aesthetic and Minimalist Design,After submitting an incorrect answer on questi...
93,KCR_P2_T2,Major Issue,"Help Recognize, Diagnose, Recover from Errors","In question 2, when the user selects incorrect..."
94,KCR_P2_T2,Minor Issue,Help and Documentation,"The ""i"" icon next to the title ""Mastering Prom..."


## 2. Aggregated Report
This section shows the consolidated issues after semantic clustering and summarization. Overlapping findings across heuristics are merged.


In [3]:
# Load all report_*.json files (excluding ranking reports)
report_files = sorted([f for f in os.listdir(REPORTS_PATH) if f.startswith("report_") and f.endswith(".json")])
all_aggregated_issues = []

for file in report_files:
    with open(os.path.join(REPORTS_PATH, file), "r") as f:
        data = json.load(f)
        for task_id, issues in data.items():
            for issue in issues:
                issue['source_file'] = file
                issue['task_id'] = task_id
                # Heuristics is a list here
                if 'heuristics' in issue:
                    issue['heuristic_list'] = ", ".join(issue['heuristics'])
                all_aggregated_issues.append(issue)

df_agg = pd.DataFrame(all_aggregated_issues)
if not df_agg.empty:
    display(Markdown(f"### Aggregated Issues for {DATA_PATH.split('/')[-2]}"))
    # Show issues grouped by Task ID for better context
    for task in df_agg['task_id'].unique():
        display(Markdown(f"#### Task: {task}"))
        task_df = df_agg[df_agg['task_id'] == task]
        display(task_df[['severity', 'heuristic_list', 'description']])
else:
    print("No aggregated reports found.")


### Aggregated Issues for KnowledgeCheckR

#### Task: KCR_P1_T1

,severity,heuristic_list,description
0,Minor Issue,"Consistency and Standards, Error Prevention, R...","Following registration, users are immediately ..."
1,Minor Issue,"Help Recognize, Diagnose, Recover from Errors,...",The generic error message for incorrect login ...
2,Minor Issue,Match between System and Real World,"The term ""CheckR App"" is used throughout the i..."
3,Minor Issue,Consistency and Standards,The login page initially displays a 'Continue'...
4,Minor Issue,Error Prevention,The login form does not provide real-time feed...


#### Task: KCR_P1_T2

,severity,heuristic_list,description
5,Minor Issue,Visibility of System Status,"When creating a new CheckR app, there is no im..."
6,Minor Issue,Visibility of System Status,"When adding a new category, there is no visual..."
7,Minor Issue,Visibility of System Status,When setting deadlines for the app or a catego...
8,Minor Issue,Match between System and Real World,"The term ""CheckR App"" might not be immediately..."
9,Minor Issue,"Match between System and Real World, Consisten...",Inconsistent terminology and date formats acro...
10,Minor Issue,Match between System and Real World,"The term ""Universal"" as a category might not b..."
11,Minor Issue,"Flexibility and Efficiency of Use, Error Preve...",Users are frustrated by the inability to delet...
12,Minor Issue,"Help Recognize, Diagnose, Recover from Errors,...",Users are frustrated by the AI question genera...
13,Minor Issue,"Consistency and Standards, Recognition Rather ...",Users are frustrated by the inconsistent and u...
14,Minor Issue,Error Prevention,"When creating multiple-choice questions, it's ..."


#### Task: KCR_P1_T3

... (3 outputs left out)

#### Task: KCR_P2_T1

,severity,heuristic_list,description
36,Minor Issue,"Match between System and Real World, Flexibili...",Users are confused and frustrated by an unexpe...
37,Minor Issue,Match between System and Real World,"The button text ""+ CheckR App"" is confusing. I..."
38,Minor Issue,"Match between System and Real World, Help and ...",Users encounter a confusing and unhelpful init...
39,Minor Issue,User Control and Freedom,The initial popup after entering the key and c...
40,Minor Issue,Consistency and Standards,"The initial page uses the term ""Enter key & Go..."
41,Minor Issue,Error Prevention,The system prompts the user to log in to acces...
42,Minor Issue,Recognition Rather than Recall,The initial landing page requires the user to ...
43,Minor Issue,Aesthetic and Minimalist Design,The initial landing page contains a lot of tex...
44,Minor Issue,"Help Recognize, Diagnose, Recover from Errors","The error message ""Login now to get access to ..."
45,Minor Issue,Help and Documentation,The initial landing page lacks clear guidance ...


#### Task: KCR_P2_T2

,severity,heuristic_list,description
46,Minor Issue,"Help Recognize, Diagnose, Recover from Errors,...",Users struggle to understand why their answers...
47,Minor Issue,Visibility of System Status,The loading animation after submitting an answ...
48,Minor Issue,Match between System and Real World,"The term ""LLMs"" might not be immediately clear..."
49,Minor Issue,Match between System and Real World,"The quiz uses terms like ""zero-shot, few-shot,..."
50,Major Issue,User Control and Freedom,"In question 2, once an answer is submitted, th..."
51,Minor Issue,Consistency and Standards,The 'Next' button changes color and shape betw...
52,Minor Issue,Consistency and Standards,The feedback messages after submitting an answ...
53,Minor Issue,Error Prevention,"In question 3, the user can drag and drop the ..."
54,Minor Issue,Recognition Rather than Recall,"In the third question, the user has to arrange..."
55,Minor Issue,Flexibility and Efficiency of Use,There is no option to quickly navigate between...


## 3. Severity Ranking
This section shows the final prioritized list of usability problems, re-ranked by an LLM based on their overall impact.


In [4]:
# Load all ranking_report_*.json files
ranking_files = sorted([f for f in os.listdir(REPORTS_PATH) if f.startswith("ranking_report_") and f.endswith(".json")])
all_ranked_issues = []

for file in ranking_files:
    with open(os.path.join(REPORTS_PATH, file), "r") as f:
        data = json.load(f)
        for task_id, issues in data.items():
            for issue in issues:
                issue['source_file'] = file
                issue['task_id'] = task_id
                all_ranked_issues.append(issue)

df_ranked = pd.DataFrame(all_ranked_issues)
if not df_ranked.empty:
    display(Markdown(f"### Final Ranked Issues for {DATA_PATH.split('/')[-2]}"))
    # Sort by Task ID if necessary, though they are usually in order
    for task_id in df_ranked['task_id'].unique():
        display(Markdown(f"#### Ranked Issues for Task: {task_id}"))
        task_df = df_ranked[df_ranked['task_id'] == task_id].reset_index(drop=True)
        # Show rank index explicitly starting from 1
        task_df.index += 1
        display(task_df[['severity', 'description', 'rank_justification']])
else:
    print("No ranking reports found.")


### Final Ranked Issues for KnowledgeCheckR

#### Ranked Issues for Task: KCR_P1_T1

,severity,description,rank_justification
1,Minor Issue,"Following registration, users are immediately ...",This issue represents a critical failure in th...
2,Minor Issue,The generic error message for incorrect login ...,This issue severely impacts the login process....
3,Minor Issue,The login form does not provide real-time feed...,The lack of real-time password validation and ...
4,Minor Issue,The login page initially displays a 'Continue'...,The inconsistent button labels on the login pa...
5,Minor Issue,"The term ""CheckR App"" is used throughout the i...",The use of the term 'CheckR App' without clear...


#### Ranked Issues for Task: KCR_P1_T2

,severity,description,rank_justification
1,Minor Issue,Users are frustrated by the inability to delet...,Users are blocked from customizing their categ...
2,Minor Issue,"When creating multiple-choice questions, it's ...",Allowing impossible values for 'Minimal correc...
3,Minor Issue,Inconsistent terminology and date formats acro...,Inconsistent terminology and date formats can ...
4,Minor Issue,Users are frustrated by the AI question genera...,Presenting the AI question generation feature ...
5,Minor Issue,Users are frustrated by the inconsistent and u...,The inconsistent and unexplained default categ...
6,Minor Issue,The platform does not provide a clear visual i...,Lack of clear visual indication of required fi...
7,Minor Issue,When attempting to delete the 'Universal' cate...,The unhelpful error message when attempting to...
8,Minor Issue,"When creating a new CheckR app, there is no im...",Lack of immediate feedback after clicking the ...
9,Minor Issue,"When adding a new category, there is no visual...",Lack of immediate visual feedback when adding ...
10,Minor Issue,When setting deadlines for the app or a catego...,Lack of clear indication that the deadline has...


#### Ranked Issues for Task: KCR_P1_T3

... (3 outputs left out)

#### Ranked Issues for Task: KCR_P2_T1

,severity,description,rank_justification
1,Minor Issue,Users are confused and frustrated by an unexpe...,This is a high severity issue because it compl...
2,Minor Issue,Users encounter a confusing and unhelpful init...,This is a high severity issue because it prese...
3,Minor Issue,The initial popup after entering the key and c...,This is a high severity issue because the conf...
4,Minor Issue,The initial landing page requires the user to ...,This is a high severity issue because it requi...
5,Minor Issue,The initial landing page lacks clear guidance ...,This is a high severity issue because it fails...
6,Minor Issue,The system prompts the user to log in to acces...,This is a medium severity issue because while ...
7,Minor Issue,"The initial page uses the term ""Enter key & Go...",This is a medium severity issue because the in...
8,Minor Issue,"The error message ""Login now to get access to ...",This is a medium severity issue because the 'e...
9,Minor Issue,"The button text ""+ CheckR App"" is confusing. I...",This is a low severity issue because while the...
10,Minor Issue,The initial landing page contains a lot of tex...,This is a low severity issue because while the...


#### Ranked Issues for Task: KCR_P2_T2

,severity,description,rank_justification
1,Minor Issue,Users struggle to understand why their answers...,Critical. Lack of clear feedback and explanati...
2,Major Issue,"In question 2, once an answer is submitted, th...",Critical. The inability to change answers viol...
3,Minor Issue,The platform provides feedback after each ques...,High. Failing to explain *why* the correct ans...
4,Minor Issue,"The quiz uses terms like ""zero-shot, few-shot,...",High. Lack of context for key terms like 'zero...
5,Minor Issue,"In question 3, the user can drag and drop the ...","High. The lack of guidance in question 3, wher..."
6,Minor Issue,"In the third question, the user has to arrange...",High. The user has to remember the order of th...
7,Minor Issue,The feedback messages after submitting an answ...,Medium. Inconsistent feedback messages create ...
8,Minor Issue,There is no option to quickly navigate between...,Medium. Inefficient navigation between questio...
9,Minor Issue,The loading animation after submitting an answ...,Medium. A vague loading animation doesn't prov...
10,Minor Issue,The drag-and-drop interface for the third ques...,Medium. Lack of clear visual cues for successf...
